In [1]:
from unstructured.partition.auto import partition
from unstructured.partition.pdf import partition_pdf
from unstructured.partition.csv import partition_csv
from unstructured.partition.xlsx import partition_xlsx
import statistics


/Users/mjmacair15/Vector_Technics_RAG_POC/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
#for unstructured document will use partition function


document1 = partition_pdf(filename="/Users/mjmacair15/Vector_Technics_RAG_POC/6b5330ba3c12ba40.pdf")
#for i, element in enumerate(document1):
#    print(f"Element {i}: {element}")

text_doc1 = "\n\n".join(str(el) for el in document1)

#determingin the chunk size based on the number of words in the document
word_counts = [len(str(el).split()) for el in document1 if str(el).strip()]
chunk_size = statistics.median(word_counts)

print(f"{len(word_counts)} elements, median length {chunk_size} words")

# Rough words->tokens conversion: 1 token ≈ 0.75 words in English,
# so words / 0.75 ≈ tokens. This is an approximation, not exact —
# exact counts need the embedding model's real tokenizer (e.g. tiktoken).

suggested_chunk_tokens_doc1 = int((chunk_size * 4) / 0.75)  # ~4 paragraphs per chunk
print(f"suggested chunk size: ~{suggested_chunk_tokens_doc1} tokens")

23 elements, median length 258 words
suggested chunk size: ~1376 tokens


<h1>trying different strategy for chunking</h1>

<h1> fixed size chunking </h1>

In [3]:
#fixed size chunking
def chunk_fixed_size(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    """Split text into fixed-size character chunks with overlap."""
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks


#run fixed size chunking on the text document
overlap_pct = 0.15 # number of characters to overlap between chunks between 15 to 20 percent
overlap_size_doc1 = int(suggested_chunk_tokens_doc1 * overlap_pct)
fixed_size_chunks_doc1 = chunk_fixed_size(text_doc1, chunk_size=suggested_chunk_tokens_doc1, overlap=overlap_size_doc1)

len(fixed_size_chunks_doc1) #43


43

<h1>sentence based chunking</h1>

In [4]:
#using the langchain text splitter for sentence based chunking
from langchain_text_splitters import NLTKTextSplitter   
splitter = NLTKTextSplitter(chunk_size=800, chunk_overlap=200)  # Adjust chunk_size and chunk_overlap as needed

sentence_chunks_langchain_doc1 = splitter.split_text(text_doc1)
print(f"Number of sentence-based chunks: {len(sentence_chunks_langchain_doc1)}")


Created a chunk of size 1464, which is longer than the specified 800


Created a chunk of size 1393, which is longer than the specified 800


Created a chunk of size 1075, which is longer than the specified 800


Created a chunk of size 3276, which is longer than the specified 800


Created a chunk of size 860, which is longer than the specified 800


...

Created a chunk of size 1533, which is longer than the specified 800


Created a chunk of size 2049, which is longer than the specified 800


Created a chunk of size 885, which is longer than the specified 800


Created a chunk of size 1770, which is longer than the specified 800


Number of sentence-based chunks: 47


<h1>RecursiveCharacterTextSplitter chunking
sentence based , paragraph based</h1>


In [24]:
#langchain text splitter for recursive character based chunking, langchain recommend default 
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1400,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""],
    )
#if we use separators=["\n\n"], it is langchain equivalent of paragraph based chunking

recursive_character_chunks_doc1 = text_splitter.split_text(text_doc1)
print(f"Number of recursive character-based chunks: {len(recursive_character_chunks_doc1)}")

Number of recursive character-based chunks: 46


<h2> using unstructured chunking by title </h2>

In [6]:
from unstructured.chunking.title import chunk_by_title
#[(2000, 200), (3000, 300), (4000, 400)]:

elements = partition_pdf(filename='6b5330ba3c12ba40.pdf')

chunk_by_title_doc1 = chunk_by_title(elements, max_characters=2500, new_after_n_chars=1000, overlap=200)
print(f'{len(chunk_by_title_doc1)} chunks')
print(chunk_by_title_doc1[0].text[:200])


28 chunks
Raphe mPhibr's Strategic Ascent: An Investment Analysis ofIndia's Leading Military Drone InnovatorI. Executive SummaryRaphe mPhibr, a prominent Noida-based drone manufacturing startup, has recentlycon


<h1>semantic based chunking</h1>
<h3>two local models</h3>

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

# Local embedding model — runs on your machine, no API key, no cost per call.
# all-MiniLM-L6-v2 is small (~80MB) and fast; downloads once, then cached.
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

splitter = SemanticChunker(embeddings)
semantic_chunks_doc1 = splitter.split_text(text_doc1)
print(f"{len(semantic_chunks_doc1)} chunks mini lm l6 v2")


embeddings_bge = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
splitter = SemanticChunker(embeddings_bge)
semantic_bge_chunks_doc1 = splitter.split_text(text_doc1)
print(f"{len(semantic_bge_chunks_doc1)} chunks bge base en v1.5")

5 chunks mini lm l6 v2


5 chunks bge base en v1.5


<h1>context based chunking</h1>

In [ ]:
import anthropic
import json

MODEL = "claude-haiku-4-5"

"""
Contextual chunking with Claude Haiku (Anthropic's "Contextual Retrieval" trick).

Takes chunks you already made with any chunker (fixed-size, sentence, paragraph,
chunk_by_title...) and asks Haiku to write a 1-2 sentence blurb that situates
each chunk inside the whole document. That blurb gets glued onto the front of
the chunk before you embed it, so a chunk that on its own reads like "Revenue
grew 40% year over year" becomes "This is from the Q3 2026 financial results
section discussing regional sales. Revenue grew 40% year over year" - much
easier for an embedding model (and a human) to retrieve correctly.

Needs: pip install anthropic (already installed in .venv)
Needs: ANTHROPIC_API_KEY set in your environment (already set - confirmed via
`echo $ANTHROPIC_API_KEY` before writing this).
"""

def add_context_to_chunks(full_document: str, chunks: list[str], client: anthropic.Anthropic = None) -> list[str]:
    """
    For every chunk, call Haiku once and prepend its answer to the chunk.

    The whole document is sent as a *cached* system block. Cache_control tells
    Anthropic "remember this exact block of text for a few minutes." Because
    `full_document` is byte-identical on every call in the loop, only the
    FIRST call pays full price to write it to cache - every call after that
    reads the same ~7k-token document back at ~10% of the input price instead
    of ~100%. Without this, contextualizing 22 chunks would mean sending the
    whole document 22 times at full price.
    """
    client = client or anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment - never hardcode the key

    contextualized = []
    for i, chunk in enumerate(chunks):
        response = client.messages.create(
            model=MODEL,
            max_tokens=300,  # a 1-2 sentence blurb never needs more than this
            system=[
                {
                    "type": "text",
                    "text": f"<document>\n{full_document}\n</document>",
                    "cache_control": {"type": "ephemeral"},  # cache this block; TTL 5 min by default
                }
            ],
            messages=[
                {
                    "role": "user",
                    "content": (
                        "Here is the chunk we want to situate within the whole document:\n"
                        f"<chunk>\n{chunk}\n</chunk>\n\n"
                        "Give a short, succinct context (1-2 sentences) that situates this "
                        "chunk within the overall document, to improve search retrieval of "
                        "the chunk. Answer with only the context, nothing else."
                    ),
                }
            ],
        )
        context_blurb = response.content[0].text.strip()
        contextualized.append(f"{context_blurb}\n\n{chunk}")

        # usage tells you whether the cache actually hit - cache_read should be
        # 0 on chunk 0 (nothing cached yet) and equal to the document's token
        # count on every chunk after that
        u = response.usage
        print(
            f"chunk {i}: cache_write={u.cache_creation_input_tokens} "
            f"cache_read={u.cache_read_input_tokens} output={u.output_tokens}"
        )

    return contextualized


def save_chunks(chunks: list[str], path: str = "contextualized_chunks_doc1.json") -> None:
    """
    Write the list of chunk strings to a JSON file. `json.dump` with no
    `indent` argument produces compact output - no pretty-printing, no
    extra whitespace - so the whole list lands on one line in the file,
    even though each chunk's own "\\n\\n" (blurb + chunk text) stays intact
    inside its string: JSON escapes real newlines *inside* a string as the
    two characters backslash-n, so they don't turn into actual line breaks
    in the file.
    """
    with open(path, "w") as f:
        json.dump(chunks, f)


def load_chunks(path: str = "contextualized_chunks_doc1.json") -> list[str]:
    """Read the JSON file back and hand you the exact same list of strings you saved."""
    with open(path) as f:
        return json.load(f)

In [9]:
#contextualized_chunks_doc1 = add_context_to_chunks(text_doc1,recursive_character_chunks_doc1) #call when needed as it is expensive to call the API

In [10]:
#save_chunks(contextualized_chunks_doc1, "text_doc1_contextualized_chunks.json")
loaded_chunks = load_chunks("text_doc1_contextualized_chunks.json")



In [11]:
print(recursive_character_chunks_doc1[0][:200])  # Print the first 200 characters of the first chunk
print(loaded_chunks[0][:200])  # Print the first 200 characters of the first contextualized chunk
#len(contextualized_chunks_doc1) #46

Raphe mPhibr's Strategic Ascent: An Investment Analysis ofIndia's Leading Military Drone InnovatorI. Executive SummaryRaphe mPhibr, a prominent Noida-based drone manufacturing startup, has recentlycon
This chunk is the Executive Summary that opens the document, providing a comprehensive overview of Raphe mPhibr's $100 million Series B funding round, the company's strategic importance to India's def


In [37]:
#lets create chunks dictionary to store the chunks for each document
chunks_dict = {"fixed_size_chunks_doc1": fixed_size_chunks_doc1,
               "sentence_chunks_langchain_doc1": sentence_chunks_langchain_doc1,
               "recursive_character_chunks_doc1": recursive_character_chunks_doc1,
               "chunk_by_title_doc1": chunk_by_title_doc1,
               "semantic_chunks_doc1": semantic_chunks_doc1,
               "semantic_bge_chunks_doc1": semantic_bge_chunks_doc1,
               "contextualized_chunks_doc1": loaded_chunks
              }

In [39]:

len(chunks_dict["recursive_character_chunks_doc1"][0])  # Example access to the first chunk of a specific document and chunking method

1398

<h1>embedding starts here</h1>

In [13]:
from sentence_transformers import SentenceTransformer, util


model_dict = {
    "all-MiniLM-L6-v2": SentenceTransformer("all-MiniLM-L6-v2"),
    "intfloat/e5-base-v2": SentenceTransformer("intfloat/e5-base-v2"),
    "BAAI/bge-m3": SentenceTransformer("BAAI/bge-m3"),
}



def embed_chunks(chunks: list[str], passage_prefix: str = "", model_name: str = "all-MiniLM-L6-v2"):
    model = model_dict.get(model_name)
    model.max_seq_length = 512
    if not model:
        raise ValueError(f"Model '{model_name}' not found in model_dict")
    prefixed_chunks = [passage_prefix + str(chunk) for chunk in chunks]
    return model.encode(prefixed_chunks, normalize_embeddings=True, convert_to_tensor=True)



In [33]:
print(chunks_dict[('recursive_character_chunks_doc1', 'all-MiniLM-L6-v2')]
                ])  # Example access to embeddings for a specific document and model

SyntaxError: closing parenthesis ']' does not match opening parenthesis '(' on line 1 (1654665809.py, line 2)

In [14]:
chunk_embeddings_dict = {}
for doc_name, chunks in chunks_dict.items():
    for model_name in model_dict.keys():
        chunk_embeddings = embed_chunks(chunks, passage_prefix="", model_name=model_name)
        chunk_embeddings_dict[(doc_name, model_name)] = chunk_embeddings
        print(f"Document: {doc_name}, Model: {model_name}, Number of embeddings: {len(chunk_embeddings)}")

Document: fixed_size_chunks_doc1, Model: all-MiniLM-L6-v2, Number of embeddings: 43


Document: fixed_size_chunks_doc1, Model: intfloat/e5-base-v2, Number of embeddings: 43


Document: fixed_size_chunks_doc1, Model: BAAI/bge-m3, Number of embeddings: 43


Document: sentence_chunks_langchain_doc1, Model: all-MiniLM-L6-v2, Number of embeddings: 47


Document: sentence_chunks_langchain_doc1, Model: intfloat/e5-base-v2, Number of embeddings: 47


...

Document: semantic_bge_chunks_doc1, Model: all-MiniLM-L6-v2, Number of embeddings: 5
Document: semantic_bge_chunks_doc1, Model: intfloat/e5-base-v2, Number of embeddings: 5


Document: semantic_bge_chunks_doc1, Model: BAAI/bge-m3, Number of embeddings: 5


Document: contextualized_chunks_doc1, Model: all-MiniLM-L6-v2, Number of embeddings: 46


Document: contextualized_chunks_doc1, Model: intfloat/e5-base-v2, Number of embeddings: 46


Document: contextualized_chunks_doc1, Model: BAAI/bge-m3, Number of embeddings: 46


In [15]:
chunk_embeddings_dict.keys()  # Example access to embeddings for a specific document and model

dict_keys([('fixed_size_chunks_doc1', 'all-MiniLM-L6-v2'), ('fixed_size_chunks_doc1', 'intfloat/e5-base-v2'), ('fixed_size_chunks_doc1', 'BAAI/bge-m3'), ('sentence_chunks_langchain_doc1', 'all-MiniLM-L6-v2'), ('sentence_chunks_langchain_doc1', 'intfloat/e5-base-v2'), ('sentence_chunks_langchain_doc1', 'BAAI/bge-m3'), ('recursive_character_chunks_doc1', 'all-MiniLM-L6-v2'), ('recursive_character_chunks_doc1', 'intfloat/e5-base-v2'), ('recursive_character_chunks_doc1', 'BAAI/bge-m3'), ('semantic_chunks_doc1', 'all-MiniLM-L6-v2'), ('semantic_chunks_doc1', 'intfloat/e5-base-v2'), ('semantic_chunks_doc1', 'BAAI/bge-m3'), ('semantic_bge_chunks_doc1', 'all-MiniLM-L6-v2'), ('semantic_bge_chunks_doc1', 'intfloat/e5-base-v2'), ('semantic_bge_chunks_doc1', 'BAAI/bge-m3'), ('contextualized_chunks_doc1', 'all-MiniLM-L6-v2'), ('contextualized_chunks_doc1', 'intfloat/e5-base-v2'), ('contextualized_chunks_doc1', 'BAAI/bge-m3')])

<h1>metrics to evaulate embedding models</h1>

In [16]:
import math
def retrieve_top_k(chunks: list[str], chunk_embeddings, query: str, k: int = 3, query_prefix: str = "", model_name: str = "all-MiniLM-L6-v2") -> list[int]:
    model = model_dict.get(model_name)
    if not model:
        raise ValueError(f"Model '{model_name}' not found in model_dict")
    query_embedding = model.encode(query_prefix + query, normalize_embeddings=True, convert_to_tensor=True)
    scores = util.cos_sim(query_embedding, chunk_embeddings)[0]
    top_k = scores.topk(min(k, len(chunks)))
    return top_k.indices.tolist()


def _metrics_from_relevance(relevant: list[int]) -> dict[str, float]:
    recall = 1 if any(relevant) else 0
    precision = sum(relevant) / len(relevant)

    first_hit_rank = next((rank for rank, is_relevant in enumerate(relevant, start=1) if is_relevant), None)
    reciprocal_rank = 1 / first_hit_rank if first_hit_rank else 0.0

    dcg = sum(rel / math.log2(rank + 1) for rank, rel in enumerate(relevant, start=1))
    ideal = sorted(relevant, reverse=True)
    idcg = sum(rel / math.log2(rank + 1) for rank, rel in enumerate(ideal, start=1))
    ndcg = dcg / idcg if idcg > 0 else 0.0

    return {"recall": recall, "precision": precision, "reciprocal_rank": reciprocal_rank, "ndcg": ndcg}

#main function to evaluate retrieval performance
def evaluate_retrieval(chunks: list[str], chunk_embeddings, eval_set: list[tuple[str, str]], k: int = 3, model_name: str = "all-MiniLM-L6-v2") -> dict[str, float, str]:
    per_question = []
    for question, answer in eval_set:
        top_k_indices = retrieve_top_k(chunks, chunk_embeddings, question, k, model_name=model_name)
        relevant = [1 if str(answer.lower()) in chunks[i].lower() else 0 for i in top_k_indices]
        per_question.append(_metrics_from_relevance(relevant))

    n = len(per_question)
    return {
        "recall@k": sum(m["recall"] for m in per_question) / n,
        "precision@k": sum(m["precision"] for m in per_question) / n,
        "mrr": sum(m["reciprocal_rank"] for m in per_question) / n,
        "ndcg@k": sum(m["ndcg"] for m in per_question) / n,
        "model_name": model_name
    }

In [17]:
from eval_set import EVAL_SET

results = evaluate_retrieval(chunks_dict["contextualized_chunks_doc1"], chunk_embeddings_dict[("contextualized_chunks_doc1", "all-MiniLM-L6-v2")], EVAL_SET, k=3)
print(results)

{'recall@k': 0.75, 'precision@k': 0.32142857142857145, 'mrr': 0.5357142857142857, 'ndcg@k': 0.5925768613504099, 'model_name': 'all-MiniLM-L6-v2'}


In [18]:
for key,value in chunk_embeddings_dict.items():
    print(key,key[0],key[1],len(value))

('fixed_size_chunks_doc1', 'all-MiniLM-L6-v2') fixed_size_chunks_doc1 all-MiniLM-L6-v2 43
('fixed_size_chunks_doc1', 'intfloat/e5-base-v2') fixed_size_chunks_doc1 intfloat/e5-base-v2 43
('fixed_size_chunks_doc1', 'BAAI/bge-m3') fixed_size_chunks_doc1 BAAI/bge-m3 43
('sentence_chunks_langchain_doc1', 'all-MiniLM-L6-v2') sentence_chunks_langchain_doc1 all-MiniLM-L6-v2 47
('sentence_chunks_langchain_doc1', 'intfloat/e5-base-v2') sentence_chunks_langchain_doc1 intfloat/e5-base-v2 47
('sentence_chunks_langchain_doc1', 'BAAI/bge-m3') sentence_chunks_langchain_doc1 BAAI/bge-m3 47
('recursive_character_chunks_doc1', 'all-MiniLM-L6-v2') recursive_character_chunks_doc1 all-MiniLM-L6-v2 46
('recursive_character_chunks_doc1', 'intfloat/e5-base-v2') recursive_character_chunks_doc1 intfloat/e5-base-v2 46
('recursive_character_chunks_doc1', 'BAAI/bge-m3') recursive_character_chunks_doc1 BAAI/bge-m3 46
('semantic_chunks_doc1', 'all-MiniLM-L6-v2') semantic_chunks_doc1 all-MiniLM-L6-v2 5
('semantic_chun

In [19]:
dense_all_results = {}  # (doc_name, model_name) -> metrics dict, feeds the dashboard's JSON export below

for embeddings,model in chunk_embeddings_dict.items():
        try:
            results = evaluate_retrieval(chunks_dict[embeddings[0]], chunk_embeddings_dict[embeddings], EVAL_SET, k=3,model_name=embeddings[1])
            dense_all_results[embeddings] = results
            print(f"Results for {list(chunks_dict.keys())[list(chunks_dict.values()).index(chunks_dict[embeddings[0]])]} with model {embeddings[1]}: {results}")
        except Exception as e:
            print(f"Error evaluating {list(chunks_dict.keys())[list(chunks_dict.values()).index(chunks_dict[embeddings[0]])]} with model {embeddings[1]}: {e}")

Results for fixed_size_chunks_doc1 with model all-MiniLM-L6-v2: {'recall@k': 0.8928571428571429, 'precision@k': 0.39285714285714285, 'mrr': 0.7261904761904762, 'ndcg@k': 0.7711482899218384, 'model_name': 'all-MiniLM-L6-v2'}


Results for fixed_size_chunks_doc1 with model intfloat/e5-base-v2: {'recall@k': 0.9642857142857143, 'precision@k': 0.4761904761904762, 'mrr': 0.7976190476190476, 'ndcg@k': 0.8333404258732156, 'model_name': 'intfloat/e5-base-v2'}


Results for fixed_size_chunks_doc1 with model BAAI/bge-m3: {'recall@k': 0.9642857142857143, 'precision@k': 0.4880952380952382, 'mrr': 0.886904761904762, 'ndcg@k': 0.9056151478250848, 'model_name': 'BAAI/bge-m3'}
Results for sentence_chunks_langchain_doc1 with model all-MiniLM-L6-v2: {'recall@k': 0.6428571428571429, 'precision@k': 0.24999999999999997, 'mrr': 0.4880952380952381, 'ndcg@k': 0.5274212843079552, 'model_name': 'all-MiniLM-L6-v2'}


Results for sentence_chunks_langchain_doc1 with model intfloat/e5-base-v2: {'recall@k': 0.8214285714285714, 'precision@k': 0.3571428571428571, 'mrr': 0.6726190476190477, 'ndcg@k': 0.7098216678538705, 'model_name': 'intfloat/e5-base-v2'}


Results for sentence_chunks_langchain_doc1 with model BAAI/bge-m3: {'recall@k': 0.9285714285714286, 'precision@k': 0.39285714285714285, 'mrr': 0.7976190476190476, 'ndcg@k': 0.831627804336741, 'model_name': 'BAAI/bge-m3'}
Results for recursive_character_chunks_doc1 with model all-MiniLM-L6-v2: {'recall@k': 0.8214285714285714, 'precision@k': 0.38095238095238093, 'mrr': 0.6666666666666667, 'ndcg@k': 0.7059927128793838, 'model_name': 'all-MiniLM-L6-v2'}


...

Results for semantic_chunks_doc1 with model BAAI/bge-m3: {'recall@k': 0.8571428571428571, 'precision@k': 0.4523809523809524, 'mrr': 0.5833333333333334, 'ndcg@k': 0.6570973469355073, 'model_name': 'BAAI/bge-m3'}
Results for semantic_bge_chunks_doc1 with model all-MiniLM-L6-v2: {'recall@k': 0.6071428571428571, 'precision@k': 0.2619047619047618, 'mrr': 0.4523809523809524, 'ndcg@k': 0.4886278677246822, 'model_name': 'all-MiniLM-L6-v2'}


Results for semantic_bge_chunks_doc1 with model intfloat/e5-base-v2: {'recall@k': 0.8571428571428571, 'precision@k': 0.3809523809523808, 'mrr': 0.5535714285714286, 'ndcg@k': 0.6270209641499812, 'model_name': 'intfloat/e5-base-v2'}


Results for semantic_bge_chunks_doc1 with model BAAI/bge-m3: {'recall@k': 0.8214285714285714, 'precision@k': 0.36904761904761896, 'mrr': 0.5892857142857144, 'ndcg@k': 0.6397789691179232, 'model_name': 'BAAI/bge-m3'}
Results for contextualized_chunks_doc1 with model all-MiniLM-L6-v2: {'recall@k': 0.75, 'precision@k': 0.32142857142857145, 'mrr': 0.5357142857142857, 'ndcg@k': 0.5925768613504099, 'model_name': 'all-MiniLM-L6-v2'}


Results for contextualized_chunks_doc1 with model intfloat/e5-base-v2: {'recall@k': 0.9285714285714286, 'precision@k': 0.4523809523809525, 'mrr': 0.75, 'ndcg@k': 0.799742473596942, 'model_name': 'intfloat/e5-base-v2'}


Results for contextualized_chunks_doc1 with model BAAI/bge-m3: {'recall@k': 1.0, 'precision@k': 0.5119047619047619, 'mrr': 0.8690476190476192, 'ndcg@k': 0.8966870549613971, 'model_name': 'BAAI/bge-m3'}


<h1>sparse embedding model</h1>

In [20]:
from sparse_embeddings import evaluate_retrieval_sparse, embed_chunks_sparse
# model loads once here (at import) instead of once per chunks_dict entry -
# don't redefine embed_chunks_sparse locally, it shadows this import and
# reloads the 2.2GB BGE-m3 model on every call

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

In [21]:
sparse_embeddings_dict = {}
sparse_all_results = {}  # doc_name -> metrics dict, feeds the dashboard's JSON export below

for key,value in chunks_dict.items():
    sparse_embeddings_dict[key] = embed_chunks_sparse(value)
    result = evaluate_retrieval_sparse(chunks_dict[key], sparse_embeddings_dict[key], EVAL_SET, k=3)
    sparse_all_results[key] = result
    print(f"Results for {key} with sparse embeddings: {result}")


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Results for fixed_size_chunks_doc1 with sparse embeddings: {'recall@k': 0.9285714285714286, 'precision@k': 0.4761904761904763, 'mrr': 0.8095238095238095, 'ndcg@k': 0.8366306158081436}


Results for sentence_chunks_langchain_doc1 with sparse embeddings: {'recall@k': 0.8571428571428571, 'precision@k': 0.3809523809523809, 'mrr': 0.738095238095238, 'ndcg@k': 0.7652020443795722}


Results for recursive_character_chunks_doc1 with sparse embeddings: {'recall@k': 0.9642857142857143, 'precision@k': 0.4761904761904763, 'mrr': 0.8511904761904762, 'ndcg@k': 0.8684196201301442}


Results for semantic_chunks_doc1 with sparse embeddings: {'recall@k': 0.8571428571428571, 'precision@k': 0.44047619047619035, 'mrr': 0.6250000000000001, 'ndcg@k': 0.6856915306106108}


Results for semantic_bge_chunks_doc1 with sparse embeddings: {'recall@k': 0.8928571428571429, 'precision@k': 0.3809523809523808, 'mrr': 0.7202380952380952, 'ndcg@k': 0.7620081808624425}


Results for contextualized_chunks_doc1 with sparse embeddings: {'recall@k': 0.9285714285714286, 'precision@k': 0.4404761904761905, 'mrr': 0.8214285714285714, 'ndcg@k': 0.8482147642791694}


<h1>hybrid search (dense + sparse, fused with RRF)</h1>

In [22]:
from hybrid_search import evaluate_retrieval as evaluate_retrieval_hybrid

# Reuses the dense (MiniLM) and sparse embeddings already computed above in
# chunk_embeddings_dict / sparse_embeddings_dict - no need to re-embed
# anything, hybrid search only fuses the two RANKINGS at query time.
hybrid_results = {}
for doc_name in chunks_dict:
    dense_embeds = chunk_embeddings_dict[(doc_name, "all-MiniLM-L6-v2")]
    sparse_weights = sparse_embeddings_dict[doc_name]
    result = evaluate_retrieval_hybrid(chunks_dict[doc_name], dense_embeds, sparse_weights, EVAL_SET, k=3)
    hybrid_results[doc_name] = result
    print(f"Results for {doc_name} with hybrid (dense+sparse): {result}")

Results for fixed_size_chunks_doc1 with hybrid (dense+sparse): {'recall@k': 0.9285714285714286, 'precision@k': 0.46428571428571436, 'mrr': 0.8511904761904762, 'ndcg@k': 0.856200380201761}


Results for sentence_chunks_langchain_doc1 with hybrid (dense+sparse): {'recall@k': 0.8571428571428571, 'precision@k': 0.3571428571428571, 'mrr': 0.6726190476190476, 'ndcg@k': 0.7220409077822536}


Results for recursive_character_chunks_doc1 with hybrid (dense+sparse): {'recall@k': 0.9642857142857143, 'precision@k': 0.4404761904761906, 'mrr': 0.8095238095238094, 'ndcg@k': 0.8510818789524485}


Results for semantic_chunks_doc1 with hybrid (dense+sparse): {'recall@k': 0.7142857142857143, 'precision@k': 0.35714285714285704, 'mrr': 0.5476190476190477, 'ndcg@k': 0.5846106087879299}


Results for semantic_bge_chunks_doc1 with hybrid (dense+sparse): {'recall@k': 0.8571428571428571, 'precision@k': 0.36904761904761896, 'mrr': 0.5773809523809526, 'ndcg@k': 0.6448781070071241}


Results for contextualized_chunks_doc1 with hybrid (dense+sparse): {'recall@k': 0.9285714285714286, 'precision@k': 0.4404761904761904, 'mrr': 0.755952380952381, 'ndcg@k': 0.793585168988735}


In [23]:
# Quick manual sanity check: does hybrid search actually surface the right
# chunk for one real question, not just the aggregate recall@k number above?
from hybrid_search import retrieve_top_k as retrieve_top_k_hybrid

doc_name = "contextualized_chunks_doc1"
question, expected_answer = EVAL_SET[0]

dense_embeds = chunk_embeddings_dict[(doc_name, "all-MiniLM-L6-v2")]
sparse_weights = sparse_embeddings_dict[doc_name]

top_k_indices = retrieve_top_k_hybrid(chunks_dict[doc_name], dense_embeds, sparse_weights, question, k=3)

print(f"Question: {question}")
print(f"Expected answer substring: {expected_answer!r}\n")
for rank, i in enumerate(top_k_indices, start=1):
    chunk_text = str(chunks_dict[doc_name][i])
    hit = expected_answer.lower() in chunk_text.lower()
    print(f"[{rank}] chunk {i} {'HIT' if hit else 'miss'}")
    print(chunk_text[:200])
    print("---")

Question: How much did Raphe mPhibr raise in its Series B round?
Expected answer substring: '$100 million'

[1] chunk 39 HIT
This chunk contains citations 2-7 from the document's Works Cited section, which are news sources documenting Raphe mPhibr's $100 million Series B funding round and its implications for India's drone 
---
[2] chunk 0 HIT
This chunk is the Executive Summary that opens the document, providing a comprehensive overview of Raphe mPhibr's $100 million Series B funding round, the company's strategic importance to India's def
---
[3] chunk 12 HIT
This chunk explains the significance of Raphe mPhibr's $100 million Series B funding round as the largest private fundraise for an Indian military drone manufacturer, and analyzes the investor confide
---


<h1>hybrid search across every chunk strategy + dense model combo</h1>

In [24]:
# The hybrid loop above (cell that prints "Results for ... with hybrid
# (dense+sparse)") only ever fuses sparse with the MiniLM dense ranking,
# because hybrid_search.py's evaluate_retrieval imports dense_embeddings.py's
# retrieve_top_k, which is hardcoded to ONE model (MiniLM) at module level -
# it has no model_name parameter.
#
# chunk_embeddings_dict, though, already has dense embeddings for ALL 3
# models (MiniLM, e5-base-v2, BGE-m3) x all 6 chunking strategies - computed
# earlier by this notebook's own model-aware embed_chunks/retrieve_top_k
# (cells b6b12be7 / b71ea6cc), not dense_embeddings.py's. So to test hybrid
# with e5/BGE-m3 too, we need a dense ranking function that takes a
# model_name - which the notebook already has (retrieve_top_k, cell b71ea6cc)
# - fused with sparse via hybrid_search.py's reciprocal_rank_fusion (that part
# IS model-agnostic already, it only looks at ranks, not which model produced
# them).

from hybrid_search import reciprocal_rank_fusion
from sparse_embeddings import retrieve_top_k as retrieve_top_k_sparse


def evaluate_hybrid(chunks, chunk_embeddings, chunk_sparse_weights, eval_set, k=3, model_name="all-MiniLM-L6-v2", rrf_k=60):
    """
    Same shape as hybrid_search.py's evaluate_retrieval, but the dense side
    can be any model in model_dict, not just MiniLM. For each eval question:
    get the FULL dense ranking (this notebook's retrieve_top_k, model-aware)
    and the FULL sparse ranking (sparse_embeddings.py's retrieve_top_k), fuse
    them with RRF, then check if the fused top-k actually contains the
    expected answer - same _metrics_from_relevance already defined above.
    """
    n = len(chunks)
    per_question = []
    for question, answer in eval_set:
        dense_ranking = retrieve_top_k(chunks, chunk_embeddings, question, k=n, model_name=model_name)
        sparse_ranking = retrieve_top_k_sparse(chunks, chunk_sparse_weights, question, k=n)
        top_k_indices = reciprocal_rank_fusion([dense_ranking, sparse_ranking], k=k, rrf_k=rrf_k)
        relevant = [1 if answer.lower() in str(chunks[i]).lower() else 0 for i in top_k_indices]
        per_question.append(_metrics_from_relevance(relevant))

    n_q = len(per_question)
    return {
        "recall@k": sum(m["recall"] for m in per_question) / n_q,
        "precision@k": sum(m["precision"] for m in per_question) / n_q,
        "mrr": sum(m["reciprocal_rank"] for m in per_question) / n_q,
        "ndcg@k": sum(m["ndcg"] for m in per_question) / n_q,
        "model_name": model_name,
    }


# chunk_embeddings_dict is already keyed (doc_name, model_name) -> dense
# embeddings for all 18 combos, so looping it directly covers every chunk
# strategy x every dense model, each fused with that doc's sparse embeddings.
hybrid_all_results = {}
for (doc_name, model_name), dense_embeds in chunk_embeddings_dict.items():
    sparse_weights = sparse_embeddings_dict[doc_name]
    result = evaluate_hybrid(chunks_dict[doc_name], dense_embeds, sparse_weights, EVAL_SET, k=3, model_name=model_name)
    hybrid_all_results[(doc_name, model_name)] = result
    print(f"Results for {doc_name} with hybrid ({model_name}+sparse): {result}")

Results for fixed_size_chunks_doc1 with hybrid (all-MiniLM-L6-v2+sparse): {'recall@k': 0.9285714285714286, 'precision@k': 0.46428571428571436, 'mrr': 0.8511904761904762, 'ndcg@k': 0.856200380201761, 'model_name': 'all-MiniLM-L6-v2'}


Results for fixed_size_chunks_doc1 with hybrid (intfloat/e5-base-v2+sparse): {'recall@k': 0.9642857142857143, 'precision@k': 0.5, 'mrr': 0.875, 'ndcg@k': 0.8999772448963251, 'model_name': 'intfloat/e5-base-v2'}


Results for fixed_size_chunks_doc1 with hybrid (BAAI/bge-m3+sparse): {'recall@k': 0.9642857142857143, 'precision@k': 0.48809523809523814, 'mrr': 0.886904761904762, 'ndcg@k': 0.9033831246091629, 'model_name': 'BAAI/bge-m3'}


Results for sentence_chunks_langchain_doc1 with hybrid (all-MiniLM-L6-v2+sparse): {'recall@k': 0.8571428571428571, 'precision@k': 0.3571428571428571, 'mrr': 0.6726190476190476, 'ndcg@k': 0.7220409077822536, 'model_name': 'all-MiniLM-L6-v2'}


Results for sentence_chunks_langchain_doc1 with hybrid (intfloat/e5-base-v2+sparse): {'recall@k': 0.9285714285714286, 'precision@k': 0.40476190476190466, 'mrr': 0.7440476190476192, 'ndcg@k': 0.785503226648345, 'model_name': 'intfloat/e5-base-v2'}


...

Results for semantic_bge_chunks_doc1 with hybrid (intfloat/e5-base-v2+sparse): {'recall@k': 0.9285714285714286, 'precision@k': 0.40476190476190466, 'mrr': 0.6011904761904763, 'ndcg@k': 0.6814395003744752, 'model_name': 'intfloat/e5-base-v2'}


Results for semantic_bge_chunks_doc1 with hybrid (BAAI/bge-m3+sparse): {'recall@k': 0.9285714285714286, 'precision@k': 0.40476190476190466, 'mrr': 0.6428571428571429, 'ndcg@k': 0.707378585572008, 'model_name': 'BAAI/bge-m3'}


Results for contextualized_chunks_doc1 with hybrid (all-MiniLM-L6-v2+sparse): {'recall@k': 0.9285714285714286, 'precision@k': 0.4404761904761904, 'mrr': 0.755952380952381, 'ndcg@k': 0.793585168988735, 'model_name': 'all-MiniLM-L6-v2'}


Results for contextualized_chunks_doc1 with hybrid (intfloat/e5-base-v2+sparse): {'recall@k': 0.9642857142857143, 'precision@k': 0.4761904761904762, 'mrr': 0.8571428571428571, 'ndcg@k': 0.8810619353201761, 'model_name': 'intfloat/e5-base-v2'}


Results for contextualized_chunks_doc1 with hybrid (BAAI/bge-m3+sparse): {'recall@k': 0.9642857142857143, 'precision@k': 0.4761904761904762, 'mrr': 0.8928571428571429, 'ndcg@k': 0.9023249578901569, 'model_name': 'BAAI/bge-m3'}


In [25]:
# Export every result dict computed above into one JSON file the dashboard
# (dashboard.html, plain HTML/JS, no build step) can fetch() and render.
# Nested as {engine: {chunk_strategy: {model: metrics}}} so the frontend's
# 3 dropdowns (chunk strategy / engine / model) are a direct 2-3 key lookup -
# no client-side searching/filtering needed.
import json as _json

dense_nested = {}
for (doc_name, model_name), result in dense_all_results.items():
    dense_nested.setdefault(doc_name, {})[model_name] = {k: v for k, v in result.items() if k != "model_name"}

hybrid_nested = {}
for (doc_name, model_name), result in hybrid_all_results.items():
    hybrid_nested.setdefault(doc_name, {})[model_name] = {k: v for k, v in result.items() if k != "model_name"}

# Sparse has no model dimension (BGE-m3 sparse only) - keyed by chunk strategy alone.
sparse_nested = {doc_name: dict(result) for doc_name, result in sparse_all_results.items()}

dashboard_data = {
    "chunk_strategies": list(chunks_dict.keys()),
    "models": list(model_dict.keys()),
    "dense": dense_nested,
    "sparse": sparse_nested,
    "hybrid": hybrid_nested,
}

with open("eval_results.json", "w") as f:
    _json.dump(dashboard_data, f, indent=2)

print(f"wrote eval_results.json: {len(dense_all_results)} dense, {len(sparse_all_results)} sparse, {len(hybrid_all_results)} hybrid results")

wrote eval_results.json: 18 dense, 6 sparse, 18 hybrid results
